In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [1]:
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/raw
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/processed

In [2]:
!hdfs dfs -ls -R /user/mahasiswa/ecommerce

drwxr-xr-x   - tara supergroup          0 2026-09-08 15:20 /user/mahasiswa/ecommerce/processed
drwxr-xr-x   - tara supergroup          0 2026-09-08 15:20 /user/mahasiswa/ecommerce/raw


In [3]:
!hdfs dfs -put transaksi_magelang.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/mahasiswa/ecommerce/raw/

In [2]:
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/raw/

Found 3 items
-rw-r--r--   1 tara supergroup     12.0 K 2026-09-08 15:21 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 tara supergroup     11.9 K 2026-09-08 15:21 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 tara supergroup     12.4 K 2026-09-08 15:21 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


In [9]:
import os
import subprocess
import pandas as pd
from IPython.display import display

# 1. Konfigurasi CLASSPATH agar Python bisa terhubung ke HDFS
hadoop_classpath = subprocess.run(["hadoop", "classpath", "--glob"], capture_output=True, text=True).stdout.strip()
os.environ["CLASSPATH"] = hadoop_classpath
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# 2. Membaca ketiga berkas dari HDFS
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/ecommerce/raw/"
df_mag = pd.read_csv(f"{path_hdfs}transaksi_magelang.csv")
df_yog = pd.read_csv(f"{path_hdfs}transaksi_yogyakarta.csv")
df_sem = pd.read_csv(f"{path_hdfs}transaksi_semarang.csv")

# 3. Menggabungkan ketiganya menjadi satu DataFrame
df_gabungan = pd.concat([df_mag, df_yog, df_sem], ignore_index=True)

# 4. Menampilkan bentuk tabel masing-masing kota (5 baris pertama)
print("=== TABEL MAGELANG ===")
display(df_mag)

print("\n=== TABEL YOGYAKARTA ===")
display(df_yog)

print("\n=== TABEL SEMARANG ===")
display(df_sem)

# 5. Membuktikan hasil gabungan (menampilkan jumlah data per kota)
print("\n=== BUKTI PENGGABUNGAN (Jumlah Transaksi per Kota) ===")
print(df_gabungan['kota'].value_counts())

=== TABEL MAGELANG ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
195,MAG-2195,2026-08-09,Makanan & Minuman,1,250000,COD,Magelang
196,MAG-2196,2026-08-28,Fashion,6,250000,COD,Magelang
197,MAG-2197,2026-08-14,Elektronik,1,250000,COD,Magelang
198,MAG-2198,2026-08-02,Makanan & Minuman,6,100000,Transfer Bank,Magelang



=== TABEL YOGYAKARTA ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,YOG-2000,2026-08-20,Fashion,5,250000,E-Wallet,Yogyakarta
1,YOG-2001,2026-08-24,Fashion,4,150000,Kartu Kredit,Yogyakarta
2,YOG-2002,2026-08-29,Rumah Tangga,4,100000,E-Wallet,Yogyakarta
3,YOG-2003,2026-08-13,Fashion,7,250000,E-Wallet,Yogyakarta
4,YOG-2004,2026-08-09,Elektronik,7,75000,COD,Yogyakarta
...,...,...,...,...,...,...,...
195,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
196,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
197,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
198,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta



=== TABEL SEMARANG ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,SEM-2000,2026-08-28,Elektronik,3,100000,E-Wallet,Semarang
1,SEM-2001,2026-08-29,Kesehatan & Kecantikan,4,25000,Transfer Bank,Semarang
2,SEM-2002,2026-08-19,Rumah Tangga,1,25000,COD,Semarang
3,SEM-2003,2026-08-02,Makanan & Minuman,2,75000,E-Wallet,Semarang
4,SEM-2004,2026-08-05,Elektronik,6,25000,Transfer Bank,Semarang
...,...,...,...,...,...,...,...
195,SEM-2195,2026-08-31,Rumah Tangga,6,75000,Transfer Bank,Semarang
196,SEM-2196,2026-08-15,Kesehatan & Kecantikan,5,75000,Transfer Bank,Semarang
197,SEM-2197,2026-08-01,Makanan & Minuman,4,75000,E-Wallet,Semarang
198,SEM-2198,2026-08-05,Fashion,7,25000,Transfer Bank,Semarang



=== BUKTI PENGGABUNGAN (Jumlah Transaksi per Kota) ===
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [6]:
!pip install fsspec pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 152.4 kB/s  0:01:01 eta 0:00:020:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fsspec]━━━━ 1/2 [fsspec]


In [11]:
# 1. Tambahkan kolom total_pendapatan (unit_terjual x harga_satuan)
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']

print("=== TABEL GABUNGAN + TOTAL PENDAPATAN ===")
display(df_gabungan)

# 2. Buat tabel ringkasan pendapatan per kota dan per kategori
ringkasan = df_gabungan.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()

print("\n=== TABEL RINGKASAN KOTA DAN KATEGORI (FULL) ===")
display(ringkasan)

# 3. Simpan sebagai CSV lokal sementara
df_gabungan.to_csv('data_gabungan_bersih.csv', index=False)
ringkasan.to_csv('ringkasan_kota_kategori.csv', index=False)
print("\n[INFO] Kedua file CSV berhasil disimpan ke disk lokal.")

=== TABEL GABUNGAN + TOTAL PENDAPATAN ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000
...,...,...,...,...,...,...,...,...
595,SEM-2195,2026-08-31,Rumah Tangga,6,75000,Transfer Bank,Semarang,450000
596,SEM-2196,2026-08-15,Kesehatan & Kecantikan,5,75000,Transfer Bank,Semarang,375000
597,SEM-2197,2026-08-01,Makanan & Minuman,4,75000,E-Wallet,Semarang,300000
598,SEM-2198,2026-08-05,Fashion,7,25000,Transfer Bank,Semarang,175000



=== TABEL RINGKASAN KOTA DAN KATEGORI (FULL) ===


,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000



[INFO] Kedua file CSV berhasil disimpan ke disk lokal.


In [5]:
# Mengunggah kedua file ke folder processed di HDFS
!hdfs dfs -put data_gabungan_bersih.csv /user/mahasiswa/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/mahasiswa/ecommerce/processed/

# Verifikasi apakah file sudah benar-benar masuk
!hdfs dfs -ls /user/mahasiswa/ecommerce/processed/

Found 2 items
-rw-r--r--   1 tara supergroup      41257 2026-09-08 15:38 /user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 tara supergroup        530 2026-09-08 15:38 /user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv
